# `SemanticAlgebra` usage examples

This page contains examples for using the `SemanticAlgebra` class.

[See full module documentation here](project:/apidocs/pogg/pogg.semantic_composition.semantic_algebra.md)

In general, most users will not need to use the methods of this class directly, as they are very generic semantic composition operations and in order to get the semantic structure you want using them you must know the exact steps for building a semantic structure from scratch.

Instead, using the methods of the classes in the semantic composition library should prove simpler.

But for those that are interested, examples of these low level operations are included below.

First, we will need the following imports:

In [1]:
import delphin.mrs as mrs
from delphin import ace
from delphin.codecs import simplemrs
import pogg.my_delphin.sementcodecs as sementcodecs
from pogg.pogg_config import POGGConfig
from pogg.semantic_composition.semantic_algebra import SemanticAlgebra

Next, we'll assume there is a `config.yml` file with the following information:

```yaml
# Grammar information
grammar_location: ./ERG_2023/erg-2023.dat
SEMI: ./ERG_2023/trunk/etc/erg.smi
```

Then we'll use this file to initialize a `POGGConfig` object and a `SemanticAlgebra` object:

In [2]:
pogg_config = POGGConfig("../config.yml")
semantic_algebra = SemanticAlgebra(pogg_config)

## `_get_slots` example

This method produces a dictionary detailing which slots are contributed by an EP (elementary predicate).

:::{info} Method details
:collapsible:
````{py:method} _get_slots(ep)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra._get_slots

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra._get_slots
```

````
:::

Per the SEM-I, the EP for `_give_v_1` has up to three semantic arguments (the giver, the thing given, and who it is given to).

```
_give_v_1 : ARG0 e, ARG1 i, ARG2 u, [ ARG3 i ].
```

Calling `_get_slots` on an EP created using this predicate label has the following result:

In [3]:
give_ep = mrs.EP("_give_v_1", "h0", pogg_config.concretize("_give_v_1"))
semantic_algebra._get_slots(give_ep)

{'ARG1': 'i2', 'ARG2': 'u3', 'ARG3': 'i4'}

## `create_base_SEMENT` example

This method creates a SEMENT with just one EP in the RELS list. The EP is created using the given predicate label and then consulting the SEMI for what slots it contributes.

:::{info} Method details}
:collapsible:
````{py:method} create_base_SEMENT(ep)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra._get_slots

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra._get_slots
```

````
:::

In [4]:
# create_base_SEMENT example
toy = semantic_algebra.create_base_SEMENT("_toy_n_1")
print(sementcodecs.encode(toy))

[ TOP: h6 INDEX: x5 RELS: < [ _toy_n_1 LBL: h6 ARG0: x5 ] > ]


## `create_CARG_SEMENT` example

This method creates a SEMENT with just one EP in the RELS list, but the EP should be one that has an argument called `CARG` for "constant argument". This is used for things like proper names and numbers. The EP is created using the given predicate label and then consulting the SEMI for what slots it contributes.

:::{info} Method details}
:collapsible:
````{py:method} create_CARG_SEMENT(ep)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.create_CARG_SEMENT

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.create_CARG_SEMENT
```

````
:::

In [5]:
# create_CARG_SEMENT example
liz = semantic_algebra.create_CARG_SEMENT("named", "Liz")
print(sementcodecs.encode(liz))

[ TOP: h8 INDEX: x7 RELS: < [ named LBL: h8 ARG0: x7 CARG: "Liz" ] > ]


## `op_non_scopal_argument_hook` example

This method performs non-scopal composition and then sets hook information of the result by passing on the hook information from the argument SEMENT.

:::{info} Method details}
:collapsible:
````{py:method} op_non_scopal_argument_hook(functor, argument, slot_label)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_non_scopal_argument_hook

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_non_scopal_argument_hook
```

````
:::

This is typically used in modification constructions (e.g. *tasty cookie*).

Below is an example of building the SEMENT for *tasty cookie* using this method.

In [6]:
# make base SEMENTs
tasty = semantic_algebra.create_base_SEMENT("_tasty_a_1")
cookie = semantic_algebra.create_base_SEMENT("_cookie_n_1")

# print the original SEMENTs
print(sementcodecs.encode(tasty, indent=True))
print(sementcodecs.encode(cookie, indent=True))

[ TOP: h11
  INDEX: e9
  RELS: < [ _tasty_a_1 LBL: h11 ARG0: e9 ARG1: u10 ] >
  SLOTS: < ARG1: u10 > ]
[ TOP: h13
  INDEX: x12
  RELS: < [ _cookie_n_1 LBL: h13 ARG0: x12 ] > ]


In [7]:
# perform composition
tasty_cookie = semantic_algebra.op_non_scopal_argument_hook(tasty, cookie, "ARG1")

# print result
print(sementcodecs.encode(tasty_cookie, indent=True))

[ TOP: h13
  INDEX: x12
  RELS: < [ _tasty_a_1 LBL: h11 ARG0: e9 ARG1: u10 ]
          [ _cookie_n_1 LBL: h13 ARG0: x12 ] >
  EQS: < h11 eq h13 u10 eq x12 > ]


Notice that the TOP and INDEX of the result come from *cookie*, hence the name `op_non_scopal_argument_hook`. Now if this SEMENT serves as an operand in further composition, such as serving as the object to *eat*, the INDEX we will have access to will be the *cookie* instance, and not the *tasty* event, which is what we want.

## `op_non_scopal_functor_hook` example

This method performs non-scopal composition and then sets hook information of the result by passing on the hook information from the functor SEMENT.

:::{info} Method details}
:collapsible:
````{py:method} op_non_scopal_functor_hook(functor, argument, slot_label)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_non_scopal_functor_hook

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_non_scopal_functor_hook
```

````
:::

This is typically used when the argument SEMENT is serving as a complement to the functor modification constructions (e.g. *tasty cookie*).

Below is an example of building the SEMENT for *eat a cookie* using this method.

In [8]:
# make base SEMENTs
eat = semantic_algebra.create_base_SEMENT("_eat_v_1")
a = semantic_algebra.create_base_SEMENT("_a_q")
cookie = semantic_algebra.create_base_SEMENT("_cookie_n_1")

# compose "a cookie" because arguments of verbs must be quantified
a_cookie = semantic_algebra.op_scopal_quantifier(a, cookie)

# print the original SEMENTs
print(sementcodecs.encode(eat, indent=True))
print(sementcodecs.encode(a_cookie, indent=True))

[ TOP: h17
  INDEX: e14
  RELS: < [ _eat_v_1 LBL: h17 ARG0: e14 ARG1: i15 ARG2: i16 ] >
  SLOTS: < ARG1: i15 ARG2: i16 > ]
[ TOP: h22
  INDEX: x18
  RELS: < [ _a_q LBL: h21 ARG0: x18 RSTR: h19 BODY: h20 ]
          [ _cookie_n_1 LBL: h24 ARG0: x23 ] >
  HCONS: < h19 qeq h24 >
  EQS: < x18 eq x23 >
  SLOTS: < BODY: h20 > ]


In [9]:
# perform composition
# plug ARG2 since that's the slot associated with the object of the verb
eat_a_cookie = semantic_algebra.op_non_scopal_functor_hook(eat, a_cookie, "ARG2")

# print result
print(sementcodecs.encode(eat_a_cookie, indent=True))

[ TOP: h17
  INDEX: e14
  RELS: < [ _eat_v_1 LBL: h17 ARG0: e14 ARG1: i15 ARG2: i16 ]
          [ _a_q LBL: h21 ARG0: x18 RSTR: h19 BODY: h20 ]
          [ _cookie_n_1 LBL: h24 ARG0: x23 ] >
  HCONS: < h19 qeq h24 >
  EQS: < x18 eq x23 h17 eq h22 i16 eq x18 >
  SLOTS: < ARG1: i15 > ]


Notice that the TOP and INDEX of the result come from *eat*, hence the name `op_non_scopal_functor_hook`.

## `op_scopal_argument_index` example

This method is analogous to `op_non_scopal_argument_hook` in that the INDEX of the result comes from the argument SEMENT, but this method is instead for scopal composition.

:::{info} Method details}
:collapsible:
````{py:method} op_scopal_argument_index(functor, argument, slot_label)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_argument_index

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_argument_index
```

````
:::

This is typically used in scopal modification constructions (e.g. *probably sleeps*).

Below is an example for building the SEMENT for *probably sleeps* using this method.

In [10]:
# make base SEMENTs
probably = semantic_algebra.create_base_SEMENT("_probable_a_1")
sleep = semantic_algebra.create_base_SEMENT("_sleep_v_1")

# print the original SEMENTs
print(sementcodecs.encode(probably, indent=True))
print(sementcodecs.encode(sleep, indent=True))

[ TOP: h27
  INDEX: i25
  RELS: < [ _probable_a_1 LBL: h27 ARG0: i25 ARG1: u26 ] >
  SLOTS: < ARG1: u26 > ]
[ TOP: h30
  INDEX: e28
  RELS: < [ _sleep_v_1 LBL: h30 ARG0: e28 ARG1: i29 ] >
  SLOTS: < ARG1: i29 > ]


In [11]:
# perform composition
probably_sleeps = semantic_algebra.op_scopal_argument_index(probably, sleep, "ARG1")

# print result
print(sementcodecs.encode(probably_sleeps, indent=True))

[ TOP: h27
  INDEX: e28
  RELS: < [ _probable_a_1 LBL: h27 ARG0: i25 ARG1: u26 ]
          [ _sleep_v_1 LBL: h30 ARG0: e28 ARG1: i29 ] >
  HCONS: < u26 qeq h30 > ]


Notice that the INDEX of the result comes from *sleeps*, but that the TOP comes from *probably*, hence the name `op_scopal_argument_index`.

## `op_scopal_functor_index` example
This method is analogous to `op_non_scopal_functor_hook` in that the INDEX of the result comes from the functor SEMENT, but this method is instead for scopal composition.

:::{info} Method details}
:collapsible:
````{py:method} op_scopal_functor_index(functor, argument, slot_label)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_functor_index

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_functor_index
```

````
:::

This is typically used in scopal constructions where the argument serves as a complement (e.g. *know the kitten snores*).

Below is an example of building a SEMENT for *know the kitten snores* using this method.

In [12]:
# make base SEMENTs
know = semantic_algebra.create_base_SEMENT("_know_v_1")
the = semantic_algebra.create_base_SEMENT("_the_q")
kitten = semantic_algebra.create_base_SEMENT("_kitten_n_1")
snore = semantic_algebra.create_base_SEMENT("_snore_v_1")

# compose "the kitten snores"
the_kitten = semantic_algebra.op_scopal_quantifier(the, kitten)
the_kitten_snores = semantic_algebra.op_non_scopal_functor_hook(snore, the_kitten, "ARG1")

# print the SEMENTs before scopal composition
print(sementcodecs.encode(know, indent=True))
print(sementcodecs.encode(the_kitten_snores, indent=True))

[ TOP: h34
  INDEX: e31
  RELS: < [ _know_v_1 LBL: h34 ARG0: e31 ARG1: u32 ARG2: i33 ] >
  SLOTS: < ARG1: u32 ARG2: i33 > ]
[ TOP: h44
  INDEX: e42
  RELS: < [ _snore_v_1 LBL: h44 ARG0: e42 ARG1: i43 ]
          [ _the_q LBL: h38 ARG0: x35 RSTR: h36 BODY: h37 ]
          [ _kitten_n_1 LBL: h41 ARG0: x40 ] >
  HCONS: < h36 qeq h41 >
  EQS: < x35 eq x40 h44 eq h39 i43 eq x35 > ]


In [13]:
# perform composition
know_the_kitten_snores = semantic_algebra.op_scopal_functor_index(know, the_kitten_snores, "ARG2")

# print result
print(sementcodecs.encode(know_the_kitten_snores, indent=True))

[ TOP: h34
  INDEX: e31
  RELS: < [ _know_v_1 LBL: h34 ARG0: e31 ARG1: u32 ARG2: i33 ]
          [ _snore_v_1 LBL: h44 ARG0: e42 ARG1: i43 ]
          [ _the_q LBL: h38 ARG0: x35 RSTR: h36 BODY: h37 ]
          [ _kitten_n_1 LBL: h41 ARG0: x40 ] >
  HCONS: < h36 qeq h41 i33 qeq h44 >
  EQS: < x35 eq x40 h44 eq h39 i43 eq x35 >
  SLOTS: < ARG1: u32 > ]


Notice that the TOP and INDEX of the result comes from *know*,hence the name `op_scopal_argument_index`.[^1]

[^1]: Technically, it could be named `op_scopal_functor_hook` because the entire hook comes from the functor, but in order to mirror the other scopal method it is named `op_scopal_functor_index` instead.

## `op_scopal_quantifier` example

This method performs scopal composition between a quantifier (e.g. *the*) and the noun (and the noun's adjuncts, if present) it quantifies. Under the hood, there are some additional steps to this operation when compared to the other scopal composition, requiring a separate method.

:::{info} Method details
:collapsible:
````{py:method} op_scopal_quantifier(functor, argument)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_quantifier

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.op_scopal_quantifier
```

````
:::

Below is an example of building a SEMENT for *the cookie* using this method.

In [14]:
the = semantic_algebra.create_base_SEMENT("_the_q")
cookie = semantic_algebra.create_base_SEMENT("_cookie_n_1")

# print the original SEMENTs
print(sementcodecs.encode(the, indent=True))
print(sementcodecs.encode(cookie, indent=True))

[ TOP: h49
  INDEX: x45
  RELS: < [ _the_q LBL: h48 ARG0: x45 RSTR: h46 BODY: h47 ] >
  SLOTS: < ARG0: x45 RSTR: h46 BODY: h47 > ]
[ TOP: h51
  INDEX: x50
  RELS: < [ _cookie_n_1 LBL: h51 ARG0: x50 ] > ]


In [15]:
# perform composition
the_cookie = semantic_algebra.op_scopal_quantifier(the, cookie)

# print result
print(sementcodecs.encode(the_cookie, indent=True))

[ TOP: h49
  INDEX: x45
  RELS: < [ _the_q LBL: h48 ARG0: x45 RSTR: h46 BODY: h47 ]
          [ _cookie_n_1 LBL: h51 ARG0: x50 ] >
  HCONS: < h46 qeq h51 >
  EQS: < x45 eq x50 >
  SLOTS: < BODY: h47 > ]


## `prepare_for_generation` example

Most of the SEMENTs above don't produce output when sent to the ERG for generation. A few requirements must be met for the ERG to generate from a semantic structure, namely:

1. The top level `INDEX` must be of type `e`
2. All entities whose intrinsic variable is of type `x` must be quantified (i.e. nouns must be quantified)
3. The `TOP` handle must be a newly introduced "GTOP" which is QEQ to the "last" TOP from composition

The method `prepare_for_generation` checks if these requirements have been met for a given SEMENT and performs additional composition operations if not to make it suitable for generation.[^bignote]

:::{info} Method details
:collapsible:
````{py:method} prepare_for_generation(sement)
:canonical: pogg.semantic_composition.semantic_algebra.SemanticAlgebra.prepare_for_generation

```{autodoc2-docstring} pogg.semantic_composition.semantic_algebra.SemanticAlgebra.prepare_for_generation
```

````
:::

[^bignote]: Regarding requirement #2, the `prepare_for_generation` method can only quantify the top level unquantified nominal phrase. For example, if you wish to generate fragments from a SEMENT that roughly corresponds to *cookies in boxes* then the SEMENT passed to `prepare_for_generation` must already have an implicit quantifier for *boxes*; this implicit quantifier should have been inserted before composing with *in*. But the head noun *cookies* may be unquantified and get "fixed" when going through `prepare_for_generation`.

This last example runs some of the SEMENTs that have been created thus far through `prepare_for_generation` and then prints the results of actually trying to generate from them.

In [16]:
# function to wrap the ACE generation code
def generate(sement):
    # encode into MRS string, since that's the format the ERG needs
    mrs_string = simplemrs.encode(sement, indent=True)

    with ace.ACEGenerator(semantic_algebra.pogg_config.grammar_location, ['-r', 'root_frag']) as generator:
        print("\n\nGENERATING FROM ... ")
        print("{}\n".format(mrs_string))
        generator_response = generator.interact(mrs_string)
        print("GENERATED RESULTS ... ")
        for r in generator_response.results():
            print(r.get('surface'))
        print("\n\n")

In [17]:
tasty_cookie_prepared = semantic_algebra.prepare_for_generation(tasty_cookie)
print("BEFORE PREPARATION...\n{}".format(sementcodecs.encode(tasty_cookie, indent=True)))
generate(tasty_cookie_prepared)

BEFORE PREPARATION...
[ TOP: h13
  INDEX: x12
  RELS: < [ _tasty_a_1 LBL: h11 ARG0: e9 ARG1: u10 ]
          [ _cookie_n_1 LBL: h13 ARG0: x12 ] >
  EQS: < h11 eq h13 u10 eq x12 > ]


GENERATING FROM ... 
[ TOP: h60
  INDEX: e57
  RELS: < [ unknown LBL: h56 ARG: x52 ARG0: e57 ]
          [ def_udef_a_q LBL: h55 ARG0: x52 RSTR: h53 BODY: h54 ]
          [ _tasty_a_1 LBL: h11 ARG0: e9 ARG1: x52 ]
          [ _cookie_n_1 LBL: h11 ARG0: x52 ] >
  HCONS: < h53 qeq h11 h60 qeq h56 > ]

GENERATED RESULTS ... 
The tasty cookie
Tasty cookies
A tasty cookie
The tasty cookies
The tasty cookies.
Tasty cookie
Tasty cookies.
The tasty cookie.
A tasty cookie.
Tasty cookie.





NOTE: 165 passive, 314 active edges in final generation chart; built 294 passives total. [10 results]
NOTE: generated 1 / 1 sentences, avg 4452k, time 0.32454s
NOTE: transfer did 188 successful unifies and 304 failed ones


In [18]:
know_the_kitten_snores_prepared = semantic_algebra.prepare_for_generation(know_the_kitten_snores)
print("BEFORE PREPARATION...\n{}".format(sementcodecs.encode(know_the_kitten_snores_prepared, indent=True)))
generate(know_the_kitten_snores_prepared)

BEFORE PREPARATION...
[ TOP: h61
  INDEX: e31
  RELS: < [ _know_v_1 LBL: h34 ARG0: e31 ARG1: u32 ARG2: h62 ]
          [ _snore_v_1 LBL: h39 ARG0: e42 ARG1: x35 ]
          [ _the_q LBL: h38 ARG0: x35 RSTR: h36 BODY: h37 ]
          [ _kitten_n_1 LBL: h41 ARG0: x35 ] >
  HCONS: < h36 qeq h41 h62 qeq h39 h61 qeq h34 >
  SLOTS: < ARG1: u32 > ]


GENERATING FROM ... 
[ TOP: h61
  INDEX: e31
  RELS: < [ _know_v_1 LBL: h34 ARG0: e31 ARG1: u32 ARG2: h62 ]
          [ _snore_v_1 LBL: h39 ARG0: e42 ARG1: x35 ]
          [ _the_q LBL: h38 ARG0: x35 RSTR: h36 BODY: h37 ]
          [ _kitten_n_1 LBL: h41 ARG0: x35 ] >
  HCONS: < h36 qeq h41 h62 qeq h39 h61 qeq h34 > ]

GENERATED RESULTS ... 
To know the kitten to snore.
To know the kittens to snore.
To know the kitten to snore
To know the kittens to snore





NOTE: 638 passive, 401 active edges in final generation chart; built 680 passives total. [4 results]
NOTE: generated 1 / 1 sentences, avg 12043k, time 0.34030s
NOTE: transfer did 315 successful unifies and 423 failed ones
